# Telecom Customer Churn Prediction

An end-to-end machine learning project for predicting telecom customer churn.

The workflow merges customer and usage data, explores churn patterns, trains a CatBoost classifier, adds behavioral features, evaluates the model with ROC-AUC, and translates the analysis into retention-oriented customer segments.

## 1. Setup

Expected local data structure:

```text
telecom/
├── Client.csv
└── Record.csv
```

The raw data is intentionally not committed to this repository.

In [ ]:
%pip install -q catboost

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42
DATA_DIR = Path("telecom")

## 2. Load and Merge the Data

In [ ]:
client_file = DATA_DIR / "Client.csv"
record_file = DATA_DIR / "Record.csv"

missing_files = [str(p) for p in [client_file, record_file] if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Add the telecom CSV files before running the notebook. Missing: "
        + ", ".join(missing_files)
    )

client = pd.read_csv(client_file)
record = pd.read_csv(record_file)

print("Client:", client.shape)
print("Record:", record.shape)

customers_df = pd.merge(client, record, on="Customer_ID")
print("Merged:", customers_df.shape)

### Recorded dataset size

The original run produced a merged dataset with **100,000 customers and 100 columns**.

In [ ]:
customers_df.head()

## 3. Exploratory Data Analysis

In [ ]:
churn_counts = customers_df["churn"].value_counts().sort_index()
churn_pct = customers_df["churn"].value_counts(normalize=True).sort_index() * 100

print(churn_counts)
print(churn_pct.round(3))

### Recorded churn distribution

- Not churned: **50,438 customers (50.438%)**
- Churned: **49,562 customers (49.562%)**

The target is therefore close to balanced.

In [ ]:
missing_percent = customers_df.isna().mean().mul(100).sort_values(ascending=False)
missing_percent[missing_percent > 0].head(25)

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=customers_df, x="churn")
plt.title("Churn Distribution")
plt.tight_layout()
plt.show()

In [ ]:
corr_with_churn = (
    customers_df.select_dtypes(include="number")
    .corr()["churn"]
    .drop("churn")
    .sort_values()
)

strongest = pd.concat([
    corr_with_churn.head(7),
    corr_with_churn.tail(7)
]).sort_values()

plt.figure(figsize=(9, 6))
strongest.plot(kind="barh")
plt.axvline(0, linewidth=1)
plt.title("Strongest Numerical Correlations with Churn")
plt.xlabel("Correlation")
plt.tight_layout()
plt.show()

Two useful EDA observations from the original run:

- Customers with older devices had a higher average churn rate.
- Customers with lower-priced handsets also showed higher churn on average.

These are associations in this dataset, not causal conclusions.

In [ ]:
print(
    customers_df.groupby("churn")[["eqpdays", "hnd_price"]]
    .mean()
    .rename(index={0: "Not churned", 1: "Churned"})
)

## 4. Train/Test Split and Preprocessing

In [ ]:
X = customers_df.drop(columns=["Customer_ID", "churn"])
y = customers_df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

def prepare_categoricals(train_part, other_part):
    train_part = train_part.copy()
    other_part = other_part.copy()

    for col in cat_cols:
        train_part[col] = train_part[col].fillna("Unknown").astype(str)
        other_part[col] = other_part[col].fillna("Unknown").astype(str)

    return train_part, other_part

X_train, X_test = prepare_categoricals(X_train, X_test)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Categorical features:", len(cat_cols))

CatBoost can handle missing numerical values directly. Categorical missing values are represented as `"Unknown"` and passed to CatBoost as categorical features.

## 5. CatBoost Baseline

In [ ]:
def make_catboost():
    return CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=0
    )

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

baseline_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train), start=1):
    X_tr = X_train.iloc[train_idx].copy()
    X_val = X_train.iloc[valid_idx].copy()
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[valid_idx]

    model = make_catboost()
    model.fit(X_tr, y_tr, cat_features=cat_cols)

    y_val_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_val_prob)
    baseline_scores.append(auc)

    print(f"Fold {fold}: AUC = {auc:.4f}")

print(f"Mean CV AUC: {np.mean(baseline_scores):.4f}")

### Recorded baseline result

The original baseline run produced a **5-fold mean validation ROC-AUC of 0.6876**.

## 6. Feature Engineering

The added features capture revenue efficiency, changes in customer behavior, device age, and overage/revenue relationships.

In [ ]:
def add_features(df):
    df = df.copy()

    df["rev_per_min"] = df["rev_Mean"] / (df["mou_Mean"] + 1)
    df["rev_ratio"] = df["change_rev"] / (df["rev_Mean"] + 1)
    df["mou_ratio"] = df["change_mou"] / (df["mou_Mean"] + 1)
    df["eqp_years"] = df["eqpdays"] / 365
    df["overage_ratio"] = df["ovrrev_Mean"] / (df["rev_Mean"] + 1)
    df["revenue_per_month"] = df["totrev"] / (df["months"] + 1)

    return df

X_train_fe = add_features(X_train)
X_test_fe = add_features(X_test)

In [ ]:
engineered_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train_fe, y_train), start=1):
    X_tr = X_train_fe.iloc[train_idx].copy()
    X_val = X_train_fe.iloc[valid_idx].copy()
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[valid_idx]

    model = make_catboost()
    model.fit(X_tr, y_tr, cat_features=cat_cols)

    y_val_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_val_prob)
    engineered_scores.append(auc)

    print(f"Fold {fold}: AUC = {auc:.4f}")

print(f"Mean CV AUC: {np.mean(engineered_scores):.4f}")

### Recorded feature-engineered CV result

The original run improved the mean validation ROC-AUC from **0.6876** to **0.6965**.

## 7. Final Model Evaluation

In [ ]:
final_model = make_catboost()
final_model.fit(X_train_fe, y_train, cat_features=cat_cols)

y_pred = final_model.predict(X_test_fe)
y_prob = final_model.predict_proba(X_test_fe)[:, 1]

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

### Recorded test results

The final feature-engineered CatBoost model achieved:

- **Test ROC-AUC: 0.6952**
- **Accuracy: 0.63695**
- **Recall for churn class: 0.65**

The goal of the model is primarily ranking churn risk rather than maximizing raw accuracy.

## 8. Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train_fe.columns,
    "Importance": final_model.get_feature_importance()
}).sort_values("Importance", ascending=False)

top_features = feature_importance.head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=top_features, x="Importance", y="Feature")
plt.title("Top CatBoost Feature Importances")
plt.tight_layout()
plt.show()

top_features

## 9. Business Segmentation and Retention Signals

The predictive model is only one part of the analysis. The following segments translate the data into practical retention targets.

In [ ]:
business_df = customers_df.copy()

business_df["customer_value"] = (
    0.4 * business_df["avgrev"]
    + 0.4 * business_df["totrev"]
    + 0.2 * business_df["months"]
)

business_df["value_segment"] = pd.qcut(
    business_df["customer_value"],
    q=4,
    labels=["Low", "Medium", "High", "VIP"]
)

business_df["old_device"] = (
    business_df["eqpdays"] > business_df["eqpdays"].median()
)

business_df["declining_customer"] = (
    (business_df["change_mou"] < 0)
    & (business_df["change_rev"] < 0)
)

business_df["inactive_lines"] = (
    business_df["uniqsubs"] - business_df["actvsubs"]
)

business_df["vip_at_risk"] = (
    (business_df["customer_value"] > business_df["customer_value"].quantile(0.75))
    & (business_df["eqpdays"] > business_df["eqpdays"].median())
)

In [ ]:
def churn_rate_by(flag):
    return pd.crosstab(
        business_df[flag],
        business_df["churn"],
        normalize="index"
    ).rename(columns={0: "Not churned", 1: "Churned"})

print("Old device:")
display(churn_rate_by("old_device"))

print("Declining usage + revenue:")
display(churn_rate_by("declining_customer"))

print("Inactive lines:")
display(churn_rate_by("inactive_lines"))

print("VIP at risk:")
display(churn_rate_by("vip_at_risk"))

### Recorded business observations

In the original analysis:

- Customers above the median device age showed **55.38% churn**, versus **43.79%** for newer-device customers.
- Customers with both declining usage and declining revenue showed **51.80% churn**, versus **48.08%** for others.
- Customers with at least one inactive line showed **56.02% churn**, versus **48.51%** for customers without inactive lines.
- The high-value + older-device segment showed **55.67% churn**, versus **48.66%** for the rest.

These segments can support targeted retention campaigns, device-upgrade offers, and proactive outreach.

## 10. Example Churn-Risk Output

In [ ]:
risk_output = pd.DataFrame({
    "Customer_ID": customers_df.loc[X_test_fe.index, "Customer_ID"],
    "Actual_Churn": y_test,
    "Predicted_Churn_Risk": y_prob
}).sort_values("Predicted_Churn_Risk", ascending=False)

risk_output.head(10)

## 11. Next Improvements

Potential extensions:

- tune CatBoost hyperparameters
- add calibration for churn probabilities
- compare CatBoost with LightGBM/XGBoost
- use SHAP for individual-customer explanations
- validate retention segments on a future time period
- optimize a decision threshold based on retention cost and customer value